# Kaggle end-to-end MOOCCubeX preprocessing and tuned BCE-SASRec

This notebook accepts the **raw MOOCCubeX-main** dataset currently attached to Kaggle. It creates memory-safe chronological Parquet splits and graph tables under `/kaggle/working`, tunes BCE-SASRec using validation NDCG@10 only, and reports train, validation and test metrics. No Google Drive or prebuilt Parquet input is required.

In [ ]:
!pip -q install ijson pyarrow duckdb tqdm psutil optuna
from pathlib import Path
print('Kaggle datasets:')
for p in Path('/kaggle/input').iterdir():print(' -',p)

## Part A — Build preprocessing outputs from raw MOOCCubeX

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, math, os, shutil, gc

import duckdb, ijson, numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
from tqdm.auto import tqdm
import torch

raw_matches=[p for p in Path('/kaggle/input').rglob('MOOCCubeX-main') if (p/'relations/user-video.json').exists()]
if not raw_matches:raise FileNotFoundError('MOOCCubeX-main with relations/user-video.json was not found under /kaggle/input')
RAW=raw_matches[0];ROOT=Path('/kaggle/working');OUT=ROOT/'processed'
SPLITS=OUT/'splits';GRAPH=OUT/'graph';REPORTS=OUT/'reports';CHECKPOINTS=OUT/'checkpoints'
for p in [OUT,SPLITS,GRAPH,REPORTS,CHECKPOINTS]:p.mkdir(parents=True,exist_ok=True)

FORCE_REBUILD=False
MIN_USER_INTERACTIONS=5
MIN_VIDEO_USERS=5
MIN_WATCH_SECONDS=5.0
SHORT_VIDEO_MAX_SECONDS=600

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('Outputs:',OUT)

## 2. Streaming JSON and watch-interval functions

In [ ]:
def first_byte(path):
 with Path(path).open('rb') as f:
  while True:
   b=f.read(1)
   if not b or not b.isspace(): return b

def stream_json(path):
 path=Path(path); b=first_byte(path)
 if b==b'[':
  with path.open('rb') as f: yield from ijson.items(f,'item')
 elif b==b'{':
  with path.open('r',encoding='utf-8') as f:
   for line in f:
    if line.strip(): yield json.loads(line)
 else: raise ValueError(path)

def merged_seconds(intervals,duration):
 clean=[]
 for a,b in intervals:
  a=max(0.0,min(float(a),duration)); b=max(0.0,min(float(b),duration))
  if b>a and math.isfinite(a) and math.isfinite(b): clean.append((a,b))
 if not clean:return 0.0
 clean.sort(); total=0.; left,right=clean[0]
 for a,b in clean[1:]:
  if a<=right:right=max(right,b)
  else:total+=right-left;left,right=a,b
 return total+right-left


## A1. Build the eligible short-video catalog

Duration is derived from the final subtitle/end timestamp. Videos between 1 second and 10 minutes are retained. The raw `video_id-ccid.txt` mapping connects behavior video IDs to video entities.

In [ ]:
mapping={}
with (RAW/'relations/video_id-ccid.txt').open('r',encoding='utf-8',errors='replace') as f:
    for line in f:
        parts=line.rstrip().split('\t')
        if len(parts)>=2:mapping[str(parts[1])]=str(parts[0])

rows=[]
for obj in tqdm(stream_json(RAW/'entities/video.json'),desc='Scanning video metadata'):
    ccid=str(obj.get('ccid') or '')
    video_id=mapping.get(ccid)
    if not video_id:continue
    starts=obj.get('start') or [];ends=obj.get('end') or [];texts=obj.get('text') or []
    values=[]
    for value in list(starts)+list(ends):
        try:
            value=float(value)
            if math.isfinite(value):values.append(value)
        except (TypeError,ValueError):pass
    duration=max(values) if values else 0.0
    rows.append({'video_id':video_id,'ccid':ccid,'duration_seconds':duration,
                 'subtitle_sentences':len(texts),'subtitle_characters':sum(len(str(x)) for x in texts),
                 'eligible_initial':bool(1<=duration<=SHORT_VIDEO_MAX_SECONDS)})
eligible=pd.DataFrame(rows)
eligible=eligible[eligible.eligible_initial].drop_duplicates('video_id').reset_index(drop=True)
duration_by_video=dict(zip(eligible.video_id.astype(str),eligible.duration_seconds.astype(float)))
ccid_by_video=dict(zip(eligible.video_id.astype(str),eligible.ccid.astype(str)))
eligible_ids=set(duration_by_video)
print({'eligible_short_videos':len(eligible),'raw_path':str(RAW)})
display(eligible.head())

## 3. Build event-level interactions

This cell rescans `user-video.json` once. It clips and merges overlapping watch intervals, computes completion ratio and a bounded engagement weight, and checkpoints the result. Rerunning will reuse the checkpoint.


In [ ]:
RAW_INTERACTIONS=CHECKPOINTS/'eligible_interactions_raw.parquet'
if FORCE_REBUILD or not RAW_INTERACTIONS.exists():
 schema=pa.schema([
  ('user_id',pa.string()),('video_id',pa.string()),('ccid',pa.string()),
  ('timestamp',pa.int64()),('last_timestamp',pa.int64()),('duration_seconds',pa.float32()),
  ('watched_seconds',pa.float32()),('playback_seconds',pa.float32()),
  ('completion_ratio',pa.float32()),('segment_count',pa.int16()),
  ('engagement_weight',pa.float32()),('positive',pa.int8())])
 writer=pq.ParquetWriter(RAW_INTERACTIONS,schema,compression='snappy')
 batch=[]; kept=invalid=0
 try:
  for obj in tqdm(stream_json(RAW/'relations/user-video.json'),desc='Preprocessing users'):
   uid=obj.get('user_id')
   if not uid:continue
   for event in obj.get('seq') or []:
    vid=str(event.get('video_id') or '')
    if vid not in eligible_ids:continue
    dur=duration_by_video[vid]; intervals=[]; playback=0.; timestamps=[]; seg_count=0
    for seg in event.get('segment') or []:
     try:
      a=float(seg['start_point']);b=float(seg['end_point']);speed=float(seg.get('speed') or 1.)
      if b<=a or speed<=0:invalid+=1;continue
      intervals.append((a,b));playback+=(b-a)/speed;seg_count+=1
      if seg.get('local_start_time') is not None:timestamps.append(int(seg['local_start_time']))
     except (KeyError,TypeError,ValueError,OverflowError):invalid+=1
    watched=merged_seconds(intervals,dur)
    if watched<MIN_WATCH_SECONDS or not timestamps:continue
    ratio=min(1.,watched/dur)
    engagement=min(1.,0.85*ratio+0.15*min(1.,seg_count/3.))
    batch.append({'user_id':str(uid),'video_id':vid,'ccid':ccid_by_video[vid],
     'timestamp':min(timestamps),'last_timestamp':max(timestamps),'duration_seconds':dur,
     'watched_seconds':watched,'playback_seconds':playback,'completion_ratio':ratio,
     'segment_count':seg_count,'engagement_weight':engagement,'positive':int(ratio>=.20)})
    kept+=1
    if len(batch)>=100000:
     writer.write_table(pa.Table.from_pylist(batch,schema=schema));batch.clear()
  if batch:writer.write_table(pa.Table.from_pylist(batch,schema=schema))
 finally:writer.close()
 json.dump({'kept_events':kept,'invalid_segments':invalid},open(REPORTS/'streaming_summary.json','w'),indent=2)
else:print('Using checkpoint:',RAW_INTERACTIONS)
print(json.load(open(REPORTS/'streaming_summary.json')))


## 4. Deduplicate and iteratively filter sparse users/videos

In [ ]:
try:
 con.close()
except Exception:
 pass
DB=Path('/kaggle/working/mooccubex_preprocessing.duckdb')
con=duckdb.connect(str(DB)); con.execute("PRAGMA threads=4"); con.execute("PRAGMA memory_limit='9GB'")
for table in ['interactions','core','core_next','ranked','train_base','valid_base','test_base','eval_users','eval_users_next','train','valid','test']:
 con.execute(f'DROP TABLE IF EXISTS {table}')
con.execute(f"CREATE TABLE interactions AS SELECT * FROM read_parquet('{RAW_INTERACTIONS.as_posix()}') WHERE positive=1")
con.execute('''CREATE OR REPLACE TABLE core AS
 SELECT user_id,video_id,any_value(ccid) AS ccid,min("timestamp") AS "timestamp",max(last_timestamp) AS last_timestamp,
 max(duration_seconds) duration_seconds,least(max(duration_seconds),sum(watched_seconds)) watched_seconds,
 sum(playback_seconds) playback_seconds,
 least(1.0,sum(watched_seconds)/nullif(max(duration_seconds),0)) completion_ratio,
 sum(segment_count) segment_count,max(engagement_weight) engagement_weight,1::TINYINT positive
 FROM interactions GROUP BY user_id,video_id''')
print('Deduplicated:',con.execute('SELECT count(*) FROM core').fetchone()[0])

for iteration in range(1,11):
 before=con.execute('SELECT count(*) FROM core').fetchone()[0]
 con.execute(f'''CREATE OR REPLACE TABLE core_next AS
  SELECT c.* FROM core c
  JOIN (SELECT user_id FROM core GROUP BY user_id HAVING count(*)>={MIN_USER_INTERACTIONS}) u USING(user_id)
  JOIN (SELECT video_id FROM core GROUP BY video_id HAVING count(DISTINCT user_id)>={MIN_VIDEO_USERS}) v USING(video_id)''')
 after=con.execute('SELECT count(*) FROM core_next').fetchone()[0]
 con.execute('DROP TABLE core');con.execute('ALTER TABLE core_next RENAME TO core')
 print(f'Iteration {iteration}: {before:,} -> {after:,}')
 if after==before:break

core_stats=con.execute('SELECT count(*) interactions,count(DISTINCT user_id) users,count(DISTINCT video_id) videos FROM core').df()
display(core_stats)


## 5. Chronological train, validation, and test splits

For every user, the newest interaction becomes test, the second newest becomes validation, and all earlier interactions become training. Validation/test items absent from training are removed to prevent item cold-start from distorting evaluation.


In [ ]:
for table in ['ranked','train_base','valid_base','test_base','eval_users','eval_users_next','train','valid','test']:
 con.execute(f'DROP TABLE IF EXISTS {table}')

con.execute('''CREATE TABLE ranked AS SELECT *,
 row_number() OVER(PARTITION BY user_id ORDER BY "timestamp" DESC,video_id) reverse_rank,
 count(*) OVER(PARTITION BY user_id) user_count FROM core''')
con.execute('CREATE TABLE train_base AS SELECT * EXCLUDE(reverse_rank,user_count) FROM ranked WHERE reverse_rank>2')
con.execute('CREATE TABLE valid_base AS SELECT * EXCLUDE(reverse_rank,user_count) FROM ranked WHERE reverse_rank=2')
con.execute('CREATE TABLE test_base AS SELECT * EXCLUDE(reverse_rank,user_count) FROM ranked WHERE reverse_rank=1')
con.execute('''CREATE TABLE eval_users AS
 SELECT user_id FROM train_base
 INTERSECT SELECT user_id FROM valid_base
 INTERSECT SELECT user_id FROM test_base''')

for iteration in range(1,11):
 old_count=con.execute('SELECT count(*) FROM eval_users').fetchone()[0]
 for table in ['train','valid','test','eval_users_next']: con.execute(f'DROP TABLE IF EXISTS {table}')
 con.execute('CREATE TABLE train AS SELECT b.* FROM train_base b JOIN eval_users USING(user_id)')
 con.execute('''CREATE TABLE valid AS SELECT b.* FROM valid_base b JOIN eval_users USING(user_id)
  JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')
 con.execute('''CREATE TABLE test AS SELECT b.* FROM test_base b JOIN eval_users USING(user_id)
  JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')
 con.execute('''CREATE TABLE eval_users_next AS
  SELECT user_id FROM train
  INTERSECT SELECT user_id FROM valid
  INTERSECT SELECT user_id FROM test''')
 new_count=con.execute('SELECT count(*) FROM eval_users_next').fetchone()[0]
 con.execute('DROP TABLE eval_users');con.execute('ALTER TABLE eval_users_next RENAME TO eval_users')
 print(f'Evaluation cleanup {iteration}: {old_count:,} -> {new_count:,} users')
 if new_count==old_count: break

# Rebuild once using the final stable user set.
for table in ['train','valid','test']: con.execute(f'DROP TABLE IF EXISTS {table}')
con.execute('CREATE TABLE train AS SELECT b.* FROM train_base b JOIN eval_users USING(user_id)')
con.execute('''CREATE TABLE valid AS SELECT b.* FROM valid_base b JOIN eval_users USING(user_id)
 JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')
con.execute('''CREATE TABLE test AS SELECT b.* FROM test_base b JOIN eval_users USING(user_id)
 JOIN (SELECT DISTINCT video_id FROM train) k USING(video_id)''')

for name in ['train','valid','test']:
 path=SPLITS/f'{name}.parquet'
 con.execute(f'''COPY (SELECT * FROM {name} ORDER BY user_id,"timestamp") TO '{path.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)''')

split_stats=con.execute('''SELECT 'train' split,count(*) interactions,count(DISTINCT user_id) users,count(DISTINCT video_id) videos FROM train
UNION ALL SELECT 'validation',count(*),count(DISTINCT user_id),count(DISTINCT video_id) FROM valid
UNION ALL SELECT 'test',count(*),count(DISTINCT user_id),count(DISTINCT video_id) FROM test''').df()
split_stats.to_csv(REPORTS/'split_statistics.csv',index=False);display(split_stats)


## 6. Create integer ID mappings

In [ ]:
users=con.execute('SELECT DISTINCT user_id FROM train ORDER BY user_id').df();users['user_idx']=np.arange(len(users),dtype=np.int64)
videos=con.execute('SELECT DISTINCT video_id,any_value(ccid) ccid FROM train GROUP BY video_id ORDER BY video_id').df();videos['video_idx']=np.arange(len(videos),dtype=np.int64)
users.to_parquet(GRAPH/'user_index.parquet',index=False);videos.to_parquet(GRAPH/'video_index.parquet',index=False)
print('Indexed users:',len(users),'videos:',len(videos))


## 7. Build filtered video–concept and course–video graph tables

In [ ]:
train_video_ids=set(videos.video_id);train_ccids=set(videos.ccid.dropna())
concept_edges=[]
with (RAW/'relations/concept-video.txt').open('r',encoding='utf-8',errors='replace') as f:
 for line in tqdm(f,desc='Concept-video edges'):
  p=line.rstrip().split('\t')
  if len(p)<2:continue
  a,b=p[0],p[1]
  concept,video_key=(a,b) if a.startswith('K_') else (b,a)
  if video_key in train_ccids:concept_edges.append((concept,video_key))
concept_edges=pd.DataFrame(concept_edges,columns=['concept_id','ccid']).drop_duplicates()
concepts=sorted(concept_edges.concept_id.unique());concept_index=pd.DataFrame({'concept_id':concepts,'concept_idx':np.arange(len(concepts),dtype=np.int64)})
concept_edges.to_parquet(GRAPH/'concept_video_edges.parquet',index=False);concept_index.to_parquet(GRAPH/'concept_index.parquet',index=False)

all_video_to_ccid={}
with (RAW/'relations/video_id-ccid.txt').open('r',encoding='utf-8',errors='replace') as f:
 for line in tqdm(f,desc='Full video ID to ccid mapping'):
  p=line.rstrip().split('\t')
  if len(p)>=2: all_video_to_ccid[p[0]]=p[1]
course_rows=[]
for course in tqdm(stream_json(RAW/'entities/course.json'),desc='Course-video graph'):
 for r in course.get('resource') or []:
  vid=str(r.get('resource_id') or '')
  ccid=all_video_to_ccid.get(vid)
  if ccid in train_ccids:course_rows.append((str(course.get('id')),vid,ccid,str(r.get('chapter') or ''),' / '.join(str(x) for x in (r.get('titles') or []) if x)))
course_video=pd.DataFrame(course_rows,columns=['course_id','video_id','ccid','chapter','titles']).drop_duplicates(['course_id','ccid'])
course_video.to_parquet(GRAPH/'course_video_edges.parquet',index=False)
print('Concept-video edges:',len(concept_edges),'Course-video edges:',len(course_video))


## 8. Save filtered video metadata and concept names

In [ ]:
video_metadata=eligible[eligible.video_id.isin(train_video_ids)].copy()
video_metadata.to_parquet(GRAPH/'video_metadata.parquet',index=False)
wanted=set(concepts);concept_rows=[]
for obj in tqdm(stream_json(RAW/'entities/concept.json'),desc='Concept metadata'):
 if obj.get('id') in wanted:concept_rows.append({'concept_id':obj.get('id'),'concept_name':obj.get('name'),'context_count':len(obj.get('context') or [])})
pd.DataFrame(concept_rows).to_parquet(GRAPH/'concept_metadata.parquet',index=False)
print('Video metadata:',len(video_metadata),'Concept metadata:',len(concept_rows))


## 9. Validate chronology and leakage

In [ ]:
checks={
 'train_before_validation':con.execute('''SELECT count(*)=0 FROM (SELECT user_id,max("timestamp") t FROM train GROUP BY user_id) a JOIN valid b USING(user_id) WHERE a.t>b."timestamp"''').fetchone()[0],
 'validation_before_test':con.execute('''SELECT count(*)=0 FROM valid a JOIN test b USING(user_id) WHERE a."timestamp">b."timestamp"''').fetchone()[0],
 'validation_items_in_train':con.execute('SELECT count(*)=0 FROM valid v WHERE NOT EXISTS (SELECT 1 FROM train t WHERE t.video_id=v.video_id)').fetchone()[0],
 'test_items_in_train':con.execute('SELECT count(*)=0 FROM test v WHERE NOT EXISTS (SELECT 1 FROM train t WHERE t.video_id=v.video_id)').fetchone()[0],
 'same_validation_test_users':con.execute('''SELECT count(*)=0 FROM
  ((SELECT user_id FROM valid EXCEPT SELECT user_id FROM test)
   UNION ALL
   (SELECT user_id FROM test EXCEPT SELECT user_id FROM valid))''').fetchone()[0],
 'all_evaluation_users_in_train':con.execute('''SELECT count(*)=0 FROM
  ((SELECT user_id FROM valid EXCEPT SELECT user_id FROM train)
   UNION ALL
   (SELECT user_id FROM test EXCEPT SELECT user_id FROM train))''').fetchone()[0],
}
validation=pd.DataFrame(checks.items(),columns=['check','passed']);validation.to_csv(REPORTS/'preprocessing_validation.csv',index=False);display(validation)
if not validation.passed.all():raise AssertionError('Preprocessing validation failed')


## 10. Save manifest and finish

In [ ]:
manifest={
 'created_at':datetime.now(timezone.utc).isoformat(),'min_user_interactions':MIN_USER_INTERACTIONS,
 'min_video_users':MIN_VIDEO_USERS,'minimum_watch_seconds':MIN_WATCH_SECONDS,
 'maximum_video_seconds':SHORT_VIDEO_MAX_SECONDS,'engagement_formula':'0.85*completion_ratio + 0.15*min(1,segment_count/3)',
 'split_method':'per-user chronological leave-two-out','split_statistics':split_stats.to_dict('records'),
 'concept_video_edges':len(concept_edges),'course_video_edges':len(course_video),
 'files':[str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()]
}
json.dump(manifest,open(REPORTS/'preprocessing_manifest.json','w'),indent=2)
print(json.dumps(manifest,indent=2))
print('Preprocessing complete:',OUT)
con.close()


## Preprocessing complete

Use `splits/train.parquet`, `splits/valid.parquet`, and `splits/test.parquet` for model training and evaluation. The `graph` directory contains the user, video and concept index mappings plus the filtered knowledge-graph edges required for explainable recommendations.


## Part B — Hyperparameter tuning and final BCE-SASRec evaluation

## 1. Imports and reproducible parameters

The maximum of 25 epochs matches the baseline experiment. Early stopping begins only after 10 epochs and stops after 5 consecutive epochs without a meaningful validation NDCG@10 improvement.

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
import copy, gc, json, math, os, random, time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

@dataclass
class Config:
    root: str = '/kaggle/input'
    seed: int = 42
    max_len: int = 50
    max_concepts_per_video: int = 12
    hidden_dim: int = 128
    transformer_layers: int = 2
    attention_heads: int = 4
    feedforward_dim: int = 512
    dropout: float = 0.10
    time_buckets: int = 32
    negatives: int = 50
    batch_size: int = 128
    eval_batch_size: int = 128
    max_epochs: int = 25
    minimum_epochs: int = 10
    early_stopping_patience: int = 5
    early_stopping_min_delta: float = 1e-4
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    concept_loss_weight: float = 0.20
    completion_loss_weight: float = 0.10
    gradient_clip: float = 5.0
    use_mixed_precision: bool = False
    ks: tuple = (5, 10, 20)
    num_workers: int = 2

CFG=Config()
ROOT=Path('/kaggle/working');PROCESSED=ROOT/'processed';SPLITS=PROCESSED/'splits';GRAPH=PROCESSED/'graph'
OUT=Path('/kaggle/working/tuned_bce_sasrec');CHECKPOINTS=OUT/'checkpoints';REPORTS=OUT/'reports';EXPLANATIONS=OUT/'explanations'
for p in [OUT,CHECKPOINTS,REPORTS,EXPLANATIONS]:p.mkdir(parents=True,exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

seed_everything(CFG.seed)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
print(json.dumps(asdict(CFG),indent=2))

## 2. Load train, validation, and test datasets

The input splits were created using per-user chronological leave-two-out. Training data supplies model fitting and normalization. Validation data controls checkpoint selection and early stopping. Test data is used only once after the best checkpoint is restored.

In [ ]:
required=[SPLITS/'train.parquet',SPLITS/'valid.parquet',SPLITS/'test.parquet',
          GRAPH/'video_metadata.parquet',GRAPH/'concept_video_edges.parquet',
          GRAPH/'video_index.parquet',GRAPH/'course_video_edges.parquet']
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError(f'Missing preprocessing outputs: {missing}')

train_df=pd.read_parquet(SPLITS/'train.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
valid_df=pd.read_parquet(SPLITS/'valid.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)
test_df=pd.read_parquet(SPLITS/'test.parquet').sort_values(['user_id','timestamp']).reset_index(drop=True)

required_columns={'user_id','video_id','timestamp','duration_seconds','watched_seconds',
                  'playback_seconds','completion_ratio','segment_count','engagement_weight'}
for name,frame in [('train',train_df),('valid',valid_df),('test',test_df)]:
    absent=required_columns-set(frame.columns)
    if absent: raise ValueError(f'{name} is missing columns: {sorted(absent)}')
    frame['user_id']=frame.user_id.astype(str); frame['video_id']=frame.video_id.astype(str)
    frame['timestamp']=pd.to_numeric(frame.timestamp,errors='coerce').fillna(0).astype('int64')

item_ids=sorted(train_df.video_id.unique()); user_ids=sorted(set(train_df.user_id)|set(valid_df.user_id)|set(test_df.user_id))
item2idx={v:i+1 for i,v in enumerate(item_ids)}; idx2item={i:v for v,i in item2idx.items()}
user2idx={u:i for i,u in enumerate(user_ids)}; idx2user={i:u for u,i in user2idx.items()}
num_items,num_users=len(item2idx),len(user2idx)

def map_frame(frame):
    x=frame[frame.video_id.isin(item2idx)].copy().reset_index(drop=True)
    x['u']=x.user_id.map(user2idx).astype('int64'); x['i']=x.video_id.map(item2idx).astype('int64')
    return x

train=map_frame(train_df);valid=map_frame(valid_df);test=map_frame(test_df)
print({'train':len(train),'validation':len(valid),'test':len(test),'users':num_users,'videos':num_items})
display(pd.DataFrame([
    ['train',len(train),train.u.nunique(),train.i.nunique()],
    ['validation',len(valid),valid.u.nunique(),valid.i.nunique()],
    ['test',len(test),test.u.nunique(),test.i.nunique()]],
    columns=['split','interactions','users','videos']))

## 3. Leakage and chronology audit

These checks must pass before training. A validation or test target may exist in another learner's training data, but it must not appear in the same learner's input history.

In [ ]:
train_max=train.groupby('u').timestamp.max()
valid_by_user=valid.set_index('u'); test_by_user=test.set_index('u')
common_users=sorted(set(train_max.index)&set(valid_by_user.index)&set(test_by_user.index))

checks={
 'train_before_or_at_validation':bool((train_max.loc[common_users].values<=valid_by_user.loc[common_users].timestamp.values).all()),
 'validation_before_or_at_test':bool((valid_by_user.loc[common_users].timestamp.values<=test_by_user.loc[common_users].timestamp.values).all()),
 'validation_items_in_training_catalog':bool(valid.i.isin(set(train.i)).all()),
 'test_items_in_training_catalog':bool(test.i.isin(set(train.i)).all()),
 'same_validation_and_test_users':set(valid.u)==set(test.u),
}
display(pd.DataFrame(checks.items(),columns=['check','passed']))
if not all(checks.values()): raise AssertionError('Leakage/chronology audit failed.')

## 4. Fit behaviour normalization on training data only

Seconds, durations, and segment counts receive `log1p` before standardization. Means and standard deviations are learned only from training interactions and then applied unchanged to validation and test.

In [ ]:
BEHAVIOUR_COLUMNS=['watched_seconds','playback_seconds','duration_seconds',
                   'completion_ratio','segment_count','engagement_weight']
LOG_BEHAVIOUR={'watched_seconds','playback_seconds','duration_seconds','segment_count'}

def raw_behaviour(frame):
    x=frame[BEHAVIOUR_COLUMNS].astype('float32').replace([np.inf,-np.inf],np.nan).fillna(0).copy()
    for c in LOG_BEHAVIOUR:x[c]=np.log1p(x[c].clip(lower=0))
    return x

train_beh_raw=raw_behaviour(train);beh_mean=train_beh_raw.mean();beh_std=train_beh_raw.std().replace(0,1).fillna(1)
def normalized_behaviour(frame):return ((raw_behaviour(frame)-beh_mean)/beh_std).astype('float32').to_numpy()
train_beh=normalized_behaviour(train);valid_beh=normalized_behaviour(valid);test_beh=normalized_behaviour(test)
normalization={'columns':BEHAVIOUR_COLUMNS,'mean':beh_mean.to_dict(),'std':beh_std.to_dict(),'log1p_columns':sorted(LOG_BEHAVIOUR)}
json.dump(normalization,open(REPORTS/'behaviour_normalization.json','w'),indent=2)
display(pd.DataFrame({'mean':beh_mean,'std':beh_std}))

## 5. Build video concept, course, and metadata inputs

Video index 0 is padding. Concept and course index 0 mean unavailable. Metadata normalization also uses the training video catalog only.

In [ ]:
video_index=pd.read_parquet(GRAPH/'video_index.parquet');video_index['video_id']=video_index.video_id.astype(str);video_index['ccid']=video_index.ccid.astype(str)
video_to_ccid=dict(zip(video_index.video_id,video_index.ccid));ccid_to_item={video_to_ccid[v]:item2idx[v] for v in item2idx if v in video_to_ccid}

cv=pd.read_parquet(GRAPH/'concept_video_edges.parquet');cv['concept_id']=cv.concept_id.astype(str);cv['ccid']=cv.ccid.astype(str)
cv=cv[cv.ccid.isin(ccid_to_item)].drop_duplicates(['ccid','concept_id'])
concept_ids=sorted(cv.concept_id.unique());concept2idx={c:i+1 for i,c in enumerate(concept_ids)};idx2concept={i:c for c,i in concept2idx.items()}
item_concepts=np.zeros((num_items+1,CFG.max_concepts_per_video),dtype=np.int64)
for ccid,g in cv.groupby('ccid'):
    ids=[concept2idx[c] for c in g.concept_id.iloc[:CFG.max_concepts_per_video]]
    item_concepts[ccid_to_item[ccid],:len(ids)]=ids

course=pd.read_parquet(GRAPH/'course_video_edges.parquet');course['video_id']=course.video_id.astype(str);course['course_id']=course.course_id.astype(str)
course=course[course.video_id.isin(item2idx)].drop_duplicates('video_id')
course_ids=sorted(course.course_id.unique());course2idx={c:i+1 for i,c in enumerate(course_ids)}
item_course=np.zeros(num_items+1,dtype=np.int64)
for row in course.itertuples():item_course[item2idx[row.video_id]]=course2idx[row.course_id]

metadata=pd.read_parquet(GRAPH/'video_metadata.parquet');metadata['video_id']=metadata.video_id.astype(str)
META_COLUMNS=['duration_seconds','subtitle_sentences','subtitle_characters','concept_count']
for c in META_COLUMNS:
    if c not in metadata:metadata[c]=0
metadata=metadata.drop_duplicates('video_id').set_index('video_id')
meta_table=pd.DataFrame(index=item_ids,columns=META_COLUMNS,dtype='float32')
for c in META_COLUMNS:meta_table[c]=pd.to_numeric(metadata.reindex(item_ids)[c],errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
for c in META_COLUMNS:meta_table[c]=np.log1p(meta_table[c].clip(lower=0))
meta_mean=meta_table.mean();meta_std=meta_table.std().replace(0,1).fillna(1)
meta_table=((meta_table-meta_mean)/meta_std).astype('float32')
item_metadata=np.zeros((num_items+1,len(META_COLUMNS)),dtype=np.float32);item_metadata[1:]=meta_table.to_numpy()

print({'concepts':len(concept_ids),'concept_video_edges':len(cv),'courses':len(course_ids),
       'videos_with_concepts':int((item_concepts!=0).any(1).sum()),'videos_with_courses':int((item_course!=0).sum())})
json.dump({'meta_columns':META_COLUMNS,'mean':meta_mean.to_dict(),'std':meta_std.to_dict()},open(REPORTS/'metadata_normalization.json','w'),indent=2)

## 6. Construct chronological histories

Validation history contains training events. Test history contains training events followed by the validation interaction. Negative sampling excludes every known train, validation, and test positive for the same learner.

In [ ]:
def build_history(frame,features):
    out={}
    for u,idx in frame.groupby('u',sort=False).groups.items():
        rows=np.asarray(list(idx));g=frame.loc[rows]
        out[int(u)]={'items':g.i.astype(int).tolist(),'times':g.timestamp.astype('int64').tolist(),
                     'behaviour':features[rows].tolist(),'completion':g.completion_ratio.astype(float).tolist()}
    return out

train_h=build_history(train,train_beh);valid_events=build_history(valid,valid_beh);test_events=build_history(test,test_beh)
eval_users=sorted(set(train_h)&set(valid_events)&set(test_events))
valid_hist={u:copy.deepcopy(train_h[u]) for u in eval_users}
test_hist={}
valid_target={u:valid_events[u]['items'][0] for u in eval_users};test_target={u:test_events[u]['items'][0] for u in eval_users}
valid_completion={u:valid_events[u]['completion'][0] for u in eval_users};test_completion={u:test_events[u]['completion'][0] for u in eval_users}
for u in eval_users:
    h=copy.deepcopy(train_h[u])
    for key in ['items','times','behaviour','completion']:h[key].append(valid_events[u][key][0])
    test_hist[u]=h

all_positive={u:set(train_h[u]['items'])|{valid_target[u],test_target[u]} for u in eval_users}
for u in train_h:
    all_positive.setdefault(u,set(train_h[u]['items']))
assert all(valid_target[u] not in valid_hist[u]['items'] for u in eval_users)
assert all(test_target[u] not in test_hist[u]['items'] for u in eval_users)
print('Leakage-free evaluation users:',len(eval_users))

## 7. Training dataset

Every training example contains only the prefix before its positive target. The target's completion ratio is an auxiliary label and is never added to the input token.

In [ ]:
def left_pad(seq,n,pad):
    # Right padding prevents fully masked attention rows under a causal mask.
    seq=list(seq)[-n:];return seq+[pad]*(n-len(seq))

class PrefixDataset(Dataset):
    def __init__(self,histories,max_len):
        self.h=histories;self.max_len=max_len
        self.examples=[(u,t) for u,h in histories.items() for t in range(1,len(h['items']))]
    def __len__(self):return len(self.examples)
    def __getitem__(self,index):
        u,t=self.examples[index];h=self.h[u]
        return (torch.tensor(u),torch.tensor(left_pad(h['items'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['times'][:t],self.max_len,0)),
                torch.tensor(left_pad(h['behaviour'][:t],self.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)),dtype=torch.float32),
                torch.tensor(h['items'][t]),torch.tensor(h['completion'][t],dtype=torch.float32))

train_dataset=PrefixDataset(train_h,CFG.max_len)
train_loader=DataLoader(train_dataset,batch_size=CFG.batch_size,shuffle=True,num_workers=CFG.num_workers,
                        pin_memory=True,persistent_workers=CFG.num_workers>0)
print('Training prefix examples:',len(train_dataset),'batches per epoch:',len(train_loader))

def sample_negatives(users,count):
    result=[]
    for u in users.tolist():
        values=[];known=all_positive[int(u)]
        while len(values)<count:
            x=random.randint(1,num_items)
            if x not in known:values.append(x)
        result.append(values)
    return torch.tensor(result,dtype=torch.long)

## 8. BCE-SASRec architecture

One causal Transformer processes unified interaction tokens. Candidate videos contain static video, concept, course, and metadata components. Learner behaviour and time are used only in historical tokens.

In [ ]:
class BCESASRec(nn.Module):
    def __init__(self,num_items,num_concepts,num_courses,item_concepts,item_course,item_metadata,cfg):
        super().__init__();self.cfg=cfg;d=cfg.hidden_dim
        self.item_emb=nn.Embedding(num_items+1,d,padding_idx=0)
        self.concept_emb=nn.Embedding(num_concepts+1,d,padding_idx=0)
        self.course_emb=nn.Embedding(num_courses+1,d,padding_idx=0)
        self.position_emb=nn.Embedding(cfg.max_len,d);self.time_emb=nn.Embedding(cfg.time_buckets,d,padding_idx=0)
        self.behaviour_mlp=nn.Sequential(nn.Linear(len(BEHAVIOUR_COLUMNS),64),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(64,d))
        self.metadata_mlp=nn.Sequential(nn.Linear(len(META_COLUMNS),64),nn.GELU(),nn.Linear(64,d))
        self.concept_query=nn.Linear(d,d,bias=False);self.concept_key=nn.Linear(d,d,bias=False)
        self.event_norm=nn.LayerNorm(d);self.candidate_norm=nn.LayerNorm(d);self.dropout=nn.Dropout(cfg.dropout)
        layer=nn.TransformerEncoderLayer(d,cfg.attention_heads,cfg.feedforward_dim,cfg.dropout,
                batch_first=True,norm_first=True,activation='gelu')
        self.transformer=nn.TransformerEncoder(layer,cfg.transformer_layers);self.output_norm=nn.LayerNorm(d)
        self.completion_head=nn.Sequential(nn.Linear(2*d,d),nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(d,1))
        self.register_buffer('item_concepts',torch.tensor(item_concepts,dtype=torch.long))
        self.register_buffer('item_course',torch.tensor(item_course,dtype=torch.long))
        self.register_buffer('item_metadata',torch.tensor(item_metadata,dtype=torch.float32))
        self.scale=math.sqrt(d)

    def concept_pool(self,item_idx,return_weights=False):
        ids=self.item_concepts[item_idx];c=self.concept_emb(ids);q=self.concept_query(self.item_emb(item_idx)).unsqueeze(-2)
        # Calculate masking and normalization in FP32. In FP16, 1e-8 becomes zero;
        # videos with no concepts would therefore divide 0 by 0 and produce NaN.
        logits=((q*self.concept_key(c)).sum(-1)/self.scale).float();mask=ids.eq(0)
        logits=logits.masked_fill(mask,-1e9);weights=torch.softmax(logits,dim=-1)
        weights=weights.masked_fill(mask,0.0)
        weights=weights/weights.sum(-1,keepdim=True).clamp_min(1.0)
        pooled=(weights.unsqueeze(-1)*c.float()).sum(-2).to(c.dtype)
        return (pooled,weights,ids) if return_weights else pooled

    def candidate(self,item_idx):
        z=self.item_emb(item_idx)+self.concept_pool(item_idx)+self.course_emb(self.item_course[item_idx])+self.metadata_mlp(self.item_metadata[item_idx])
        return self.candidate_norm(z)

    def time_bucket(self,times):
        gap=torch.zeros_like(times);valid=(times[:,1:]>0)&(times[:,:-1]>0)
        delta=(times[:,1:]-times[:,:-1]).clamp_min(0)
        gap[:,1:]=torch.where(valid,delta,torch.zeros_like(delta))
        bucket=torch.floor(torch.log2(gap.float()+1)).long()+1
        return bucket.clamp(0,self.cfg.time_buckets-1).masked_fill(times.eq(0),0)

    def encode(self,seq,times,behaviour):
        pos=torch.arange(self.cfg.max_len,device=seq.device).unsqueeze(0)
        static=self.candidate(seq);x=static+self.behaviour_mlp(behaviour)+self.time_emb(self.time_bucket(times))+self.position_emb(pos)
        padding=seq.eq(0);x=self.dropout(self.event_norm(x));x=x.masked_fill(padding.unsqueeze(-1),0.0)
        causal=torch.triu(torch.ones(self.cfg.max_len,self.cfg.max_len,device=seq.device,dtype=torch.bool),1)
        x=self.transformer(x,mask=causal,src_key_padding_mask=padding)
        last_index=seq.ne(0).sum(1).clamp_min(1)-1
        last=x[torch.arange(len(seq),device=seq.device),last_index]
        return self.output_norm(last)

    def sampled_logits(self,h,candidates):return (h.unsqueeze(1)*self.candidate(candidates)).sum(-1)/self.scale
    def all_candidate_embeddings(self):return self.candidate(torch.arange(1,self.item_emb.num_embeddings,device=self.item_emb.weight.device))
    def completion(self,h,item):return torch.sigmoid(self.completion_head(torch.cat([h,self.candidate(item)],-1))).squeeze(-1)

model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,CFG).to(device)
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 9. Correct recommendation metrics

There is one relevant next video for each learner. Consequently, `Precision@K = Recall@K / K`, and the maximum possible F1@10 is 0.1818. NDCG@10 is the primary selection metric.

In [ ]:
def calculate_metrics(ranks,topk_items,item_popularity,num_items,ks=(5,10,20)):
    ranks=np.asarray(ranks,dtype=np.int64);out={'Accuracy@1':float(np.mean(ranks==1)),'MRR':float(np.mean(1/ranks)),
        'MeanRank':float(np.mean(ranks)),'MedianRank':float(np.median(ranks))}
    for k in ks:
        hit=ranks<=k;recall=float(hit.mean());precision=recall/k
        out[f'Precision@{k}']=precision;out[f'Recall@{k}']=recall
        out[f'F1@{k}']=0.0 if recall==0 else float(2*precision*recall/(precision+recall))
        out[f'NDCG@{k}']=float(np.mean(np.where(hit,1/np.log2(ranks+1),0)))
        out[f'MAP@{k}']=float(np.mean(np.where(hit,1/ranks,0)))
    rec=np.asarray(topk_items);out['CatalogCoverage@10']=float(len(np.unique(rec))/num_items)
    total=sum(item_popularity.values());probs=np.array([item_popularity.get(int(i),0.5)/total for i in rec.ravel()])
    out['Novelty@10']=float(np.mean(-np.log2(np.clip(probs,1e-12,None))))
    return out

item_popularity=train.i.value_counts().to_dict()

def tensors_for_users(histories,users):
    seq=torch.tensor([left_pad(histories[u]['items'],CFG.max_len,0) for u in users],device=device)
    times=torch.tensor([left_pad(histories[u]['times'],CFG.max_len,0) for u in users],device=device)
    beh=torch.tensor([left_pad(histories[u]['behaviour'],CFG.max_len,[0.0]*len(BEHAVIOUR_COLUMNS)) for u in users],dtype=torch.float32,device=device)
    return seq,times,beh

@torch.no_grad()
def evaluate(model,histories,targets,target_completion,users,description):
    model.eval();candidate_z=model.all_candidate_embeddings();ranks=[];top_items=[];loss_sum=0.;n=0;completion_errors=[]
    for start in tqdm(range(0,len(users),CFG.eval_batch_size),desc=description,leave=False):
        us=users[start:start+CFG.eval_batch_size];seq,times,beh=tensors_for_users(histories,us)
        h=model.encode(seq,times,beh);scores=(h@candidate_z.T)/model.scale
        if not torch.isfinite(scores).all():
            raise FloatingPointError('Non-finite evaluation scores detected; metrics were not calculated.')
        target=torch.tensor([targets[u]-1 for u in us],device=device)
        for row,u in enumerate(us):
            seen=set(histories[u]['items']);seen.discard(targets[u])
            if seen:scores[row,torch.tensor([i-1 for i in seen],device=device)]=torch.finfo(scores.dtype).min
        loss_sum+=F.cross_entropy(scores,target,reduction='sum').item();n+=len(us)
        target_scores=scores[torch.arange(len(us),device=device),target]
        ranks.extend(((scores>target_scores.unsqueeze(1)).sum(1)+1).cpu().tolist())
        top_items.extend((torch.topk(scores,k=10,dim=1).indices+1).cpu().tolist())
        completion_pred=model.completion(h,target+1).float()
        if not torch.isfinite(completion_pred).all():
            raise FloatingPointError('Non-finite completion predictions detected.')
        completion_pred=completion_pred.cpu().numpy()
        completion_true=np.asarray([target_completion[u] for u in us],dtype=np.float32)
        completion_errors.extend((completion_pred-completion_true).tolist())
    metrics=calculate_metrics(ranks,top_items,item_popularity,num_items,CFG.ks);metrics['Loss']=loss_sum/n
    err=np.asarray(completion_errors);metrics['CompletionMAE']=float(np.mean(np.abs(err)));metrics['CompletionRMSE']=float(np.sqrt(np.mean(err**2)))
    concept_recalls=[]
    for u,recs in zip(users,top_items):
        true=set(item_concepts[targets[u]])-{0};pred=set(item_concepts[np.asarray(recs)].ravel())-{0}
        if true:concept_recalls.append(len(true&pred)/len(true))
    metrics['ConceptRecall@10']=float(np.mean(concept_recalls)) if concept_recalls else float('nan')
    return metrics,ranks,top_items

## 10. Create a comparable training-evaluation split

For training metrics, the final training event is the target and earlier training events form its history. This is an in-sample diagnostic and must not be interpreted as generalization performance.

In [ ]:
train_eval_users=sorted(u for u,h in train_h.items() if len(h['items'])>=2)
train_eval_hist={};train_eval_target={};train_eval_completion={}
for u in train_eval_users:
    h=train_h[u]
    train_eval_hist[u]={k:list(v[:-1]) for k,v in h.items()}
    train_eval_target[u]=h['items'][-1]
    train_eval_completion[u]=h['completion'][-1]
print({'training_evaluation_users':len(train_eval_users),'validation_users':len(eval_users),'test_users':len(eval_users)})

## 11. Training function

Each trial uses sampled negatives for optimization and full-catalog validation for selection. It never evaluates the test targets.

In [ ]:
def concept_pair_loss(model,h,pos_items):
    ids=model.item_concepts[pos_items];mask=ids.ne(0);has=mask.any(1)
    if not has.any():return h.sum()*0
    first=mask.float().argmax(1);positive=ids[torch.arange(len(ids),device=device),first]
    negative=torch.randint(1,model.concept_emb.num_embeddings,(len(ids),),device=device)
    negative=torch.where(negative.eq(positive),(negative%(model.concept_emb.num_embeddings-1))+1,negative)
    ps=(h*model.concept_emb(positive)).sum(-1)/model.scale;ns=(h*model.concept_emb(negative)).sum(-1)/model.scale
    return -F.logsigmoid(ps[has]-ns[has]).mean()

def fit_model(cfg,name,max_epochs,trial=None):
    seed_everything(cfg.seed)
    model=BCESASRec(num_items,len(concept_ids),len(course_ids),item_concepts,item_course,item_metadata,cfg).to(device)
    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
    scaler=torch.amp.GradScaler('cuda',enabled=cfg.use_mixed_precision and device.type=='cuda')
    best=-float('inf');bad=0;history=[];path=CHECKPOINTS/f'{name}_best.pt'
    for epoch in range(1,max_epochs+1):
        started=time.time();model.train();sums=defaultdict(float);examples=0
        for users,seq,times,behaviour,pos,completion in tqdm(train_loader,desc=f'{name} {epoch:02d}/{max_epochs}',leave=False):
            users,seq,times,behaviour,pos,completion=[x.to(device,non_blocking=True) for x in [users,seq,times,behaviour,pos,completion]]
            negatives=sample_negatives(users.cpu(),cfg.negatives).to(device);candidates=torch.cat([pos[:,None],negatives],1)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type,enabled=cfg.use_mixed_precision and device.type=='cuda'):
                h=model.encode(seq,times,behaviour);logits=model.sampled_logits(h,candidates)
                ranking=F.cross_entropy(logits,torch.zeros(len(seq),dtype=torch.long,device=device),label_smoothing=.03)
                concept=concept_pair_loss(model,h,pos);pred=model.completion(h,pos)
                completion_loss=F.mse_loss(pred,completion.clamp(0,1))
                total=ranking+cfg.concept_loss_weight*concept+cfg.completion_loss_weight*completion_loss
            if not torch.isfinite(total):raise FloatingPointError(f'Non-finite loss: {name}, epoch {epoch}')
            scaler.scale(total).backward();scaler.unscale_(optimizer)
            grad=nn.utils.clip_grad_norm_(model.parameters(),cfg.gradient_clip)
            if not torch.isfinite(grad):raise FloatingPointError(f'Non-finite gradient: {name}, epoch {epoch}')
            scaler.step(optimizer);scaler.update();bs=len(seq);examples+=bs
            for k,v in [('TrainLoss',total),('RankingLoss',ranking),('ConceptLoss',concept),('CompletionLoss',completion_loss)]:sums[k]+=float(v.detach())*bs
        val,_,_=evaluate(model,valid_hist,valid_target,valid_completion,eval_users,'Validation')
        row={'Epoch':epoch,**{k:v/examples for k,v in sums.items()},**{f'Val_{k}':v for k,v in val.items()},'Seconds':time.time()-started}
        history.append(row);score=val['NDCG@10']
        print(f"{name} epoch {epoch}: loss={row['TrainLoss']:.4f}, val NDCG@10={score:.4f}, val Recall@10={val['Recall@10']:.4f}")
        if score>best+cfg.early_stopping_min_delta:
            best=score;bad=0;torch.save({'model_state':model.state_dict(),'epoch':epoch,'validation_metrics':val,'config':asdict(cfg)},path)
        else:bad+=1
        if trial is not None:
            trial.report(score,epoch)
            if trial.should_prune():raise optuna.TrialPruned()
        if epoch>=cfg.minimum_epochs and bad>=cfg.early_stopping_patience:break
    pd.DataFrame(history).to_csv(REPORTS/f'{name}_epochs.csv',index=False)
    saved=torch.load(path,map_location=device);model.load_state_dict(saved['model_state'])
    return model,saved,pd.DataFrame(history)

## 12. Validation-only hyperparameter search

Eight trials balance tuning quality with T4 runtime. Increase `N_TRIALS` only if additional compute is available.

In [ ]:
import optuna
N_TRIALS=8
def objective(trial):
    cfg=copy.deepcopy(CFG)
    cfg.hidden_dim=trial.suggest_categorical('hidden_dim',[128,256])
    cfg.transformer_layers=trial.suggest_int('transformer_layers',2,3)
    cfg.attention_heads=trial.suggest_categorical('attention_heads',[4,8])
    cfg.feedforward_dim=trial.suggest_categorical('feedforward_dim',[512,1024])
    cfg.dropout=trial.suggest_float('dropout',.10,.25)
    cfg.learning_rate=trial.suggest_categorical('learning_rate',[1e-4,3e-4,5e-4])
    cfg.weight_decay=trial.suggest_categorical('weight_decay',[1e-5,1e-4])
    cfg.negatives=trial.suggest_categorical('negatives',[100,200])
    cfg.concept_loss_weight=trial.suggest_float('concept_loss_weight',.10,.25)
    cfg.completion_loss_weight=trial.suggest_categorical('completion_loss_weight',[.03,.05,.10])
    cfg.minimum_epochs=5;cfg.early_stopping_patience=3
    model,saved,_=fit_model(cfg,f'trial_{trial.number:02d}',8,trial)
    score=saved['validation_metrics']['NDCG@10'];del model;gc.collect();torch.cuda.empty_cache();return score

study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=CFG.seed),pruner=optuna.pruners.MedianPruner(n_startup_trials=3,n_warmup_steps=3))
study.optimize(objective,n_trials=N_TRIALS)
print('Best validation NDCG@10:',study.best_value);print('Best parameters:',study.best_params)
study.trials_dataframe().to_csv(REPORTS/'hyperparameter_trials.csv',index=False)

## 13. Train the selected configuration for up to 25 epochs

In [ ]:
BEST=copy.deepcopy(CFG)
for key,value in study.best_params.items():setattr(BEST,key,value)
BEST.max_epochs=25;BEST.minimum_epochs=10;BEST.early_stopping_patience=5;BEST.early_stopping_min_delta=1e-4
tuned_model,saved,history=fit_model(BEST,'Tuned_BCE_SASRec',BEST.max_epochs)
print('Best epoch:',saved['epoch']);print('Best validation NDCG@10:',saved['validation_metrics']['NDCG@10'])

## 14. Final train, validation and test metric table

The test split is accessed here for the first and only time after tuning and checkpoint selection are complete.

In [ ]:
train_metrics,_,_=evaluate(tuned_model,train_eval_hist,train_eval_target,train_eval_completion,train_eval_users,'Train diagnostic')
validation_metrics,_,_=evaluate(tuned_model,valid_hist,valid_target,valid_completion,eval_users,'Validation final')
test_metrics,test_ranks,test_top10=evaluate(tuned_model,test_hist,test_target,test_completion,eval_users,'Test final')

requested=['Accuracy@1','Recall@10','Recall@20','ConceptRecall@10']
summary=pd.DataFrame({
    'Metric':requested,
    'Train':[train_metrics[m] for m in requested],
    'Validation':[validation_metrics[m] for m in requested],
    'Test':[test_metrics[m] for m in requested],
})
display(summary.style.format({'Train':'{:.4f}','Validation':'{:.4f}','Test':'{:.4f}'}))
summary.to_csv(REPORTS/'train_validation_test_requested_metrics.csv',index=False)

all_metrics=pd.DataFrame([{'Split':'Train',**train_metrics},{'Split':'Validation',**validation_metrics},{'Split':'Test',**test_metrics}])
display(all_metrics);all_metrics.to_csv(REPORTS/'train_validation_test_all_metrics.csv',index=False)
json.dump({'model':'Tuned BCE-SASRec','best_epoch':saved['epoch'],'best_parameters':study.best_params,'config':asdict(BEST),'train_metrics':train_metrics,'validation_metrics':validation_metrics,'test_metrics':test_metrics},open(REPORTS/'tuned_experiment_manifest.json','w'),indent=2)
print('Saved outputs to:',REPORTS)

## 15. Plot metrics and package outputs

In [ ]:
h=history;fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].plot(h.Epoch,h.TrainLoss,label='Train loss');ax[0].plot(h.Epoch,h.Val_Loss,label='Validation loss');ax[0].legend();ax[0].grid(alpha=.25)
ax[1].plot(h.Epoch,h['Val_NDCG@10'],label='NDCG@10');ax[1].plot(h.Epoch,h['Val_Recall@10'],label='Recall@10');ax[1].legend();ax[1].grid(alpha=.25)
fig.tight_layout();fig.savefig(REPORTS/'tuned_training_curves.png',dpi=180);plt.show()
import shutil
zip_path=shutil.make_archive('/kaggle/working/Tuned_BCE_SASRec_outputs','zip',root_dir=str(OUT.parent),base_dir=OUT.name)
print('ZIP:',zip_path)